In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

2025-12-06 15:07:27.886344: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-06 15:07:27.889698: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
import pandas as pd
import numpy as np

In [3]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=3:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=4)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

2025-12-06 15:07:30,770 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2025-12-06 15:07:30,771 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2025-12-06 15:07:30,772 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2025-12-06 15:07:30,773 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB


In [4]:
DATA_ROOT="/home/mcn26/project_pi_skr2/shared/tabula_data"
simpath="simulated/shendure_pow_analysis/sim_with_orthos_20251203"
dat=scm.scMPRA_data.from_parquet(f"{DATA_ROOT}/{simpath}/scMPRA/0.scmpra").data
o0=scm.ortho.load(client,f"{DATA_ROOT}/{simpath}/orthos","0")
mats=o0.by_cell_type_design["Cardiomyocytes"].result()

In [7]:
import re

def _extract_square(s):
    """
    Utility function which extracts the contents of [T. ] or [ ] brackets from  string `s`
    Assumes one pair of brackets
    If no brackets are found, returns `s` untouched.
    """
    tm = re.search(r"\[T\.(.*?)\]", s)
    m = re.search(r"\[(.*?)\]", s)
    if tm:
        return tm.group(1)
    elif m:
        return m.group(1)
    else:
        return s
    

def _matricies_to_order(matricies):
    """
    Helper function. Extracts column index order from matricies for use elsewhere.
    Replaces `Intercept` with `reference`.
    """
    zi_idx=matricies["zi_regressors"].columns.to_list()
    zi_idx=[_extract_square(s) if s !="Intercept" else "reference" for s in zi_idx]

    nb_idx=mats["nb_regressors"].columns.to_list()
    nb_idx=[_extract_square(s) if s !="Intercept" else "reference" for s in nb_idx]

    return {'zi_idx':zi_idx,'nb_idx':nb_idx}

def _mom_from_training_data(data,split,subset,indicies):
    """
    Helper function implementing warm start method of moments for parameter initalization.
    See [[Fixing zinb initialization]] and [[LFC is beta]] for math.

    `indicies` are the return of of `_matricies_to_order`
    """
    
    anti=scm.anti_split(split)

    #clean up raw training data
    raw=data[["rep_id","cell_type","cre_id","umis_mpra_bc"]]
    raw=data[data[split]==subset]
    raw=raw.drop(columns=split)

    #collapse to summary statistics
    nb_stats = (
        raw.groupby(anti)
        .agg(
            mean_umis_mpra_bc=('umis_mpra_bc', 'mean'),
            var_umis_mpra_bc=('umis_mpra_bc', 'var')
        )
    )

    #get that set which are valid nb
    nb_stats["valid_nb"]=nb_stats["var_umis_mpra_bc"] > nb_stats["mean_umis_mpra_bc"]

    # compute rep-level counts including zeros
    zi_stats = (
        raw.groupby(['rep_id', anti])
        .agg(
            n=('umis_mpra_bc', 'count'),
            n_zero=('umis_mpra_bc', lambda x: (x == 0).sum())
        )
    )

    
    # merge global mean/var into rep-level
    zi_stats = zi_stats.reset_index().merge(nb_stats, on=anti, how='left')

    #so a row in zi_stats represents 'for replicate [rep_id] 
    #we see [cre_id] [n] times, of which [n_zero] data points are zero
    #that cre, across all reps, has a mean of [mean_umis_mpra_bc] and a 
    #variance of [var_umis_mpra_bc].' 
    
    
    
    zi_stats["gross_zero_prop"]=zi_stats["n_zero"]/zi_stats["n"]
    
    #We now want to figure out how many zeros are expected from the nb portion
    
    
    #compute nb params
    zi_stats["p"]=zi_stats["mean_umis_mpra_bc"]/zi_stats["var_umis_mpra_bc"]
    zi_stats["r"]=zi_stats["mean_umis_mpra_bc"]**2 / (zi_stats["var_umis_mpra_bc"] - zi_stats["mean_umis_mpra_bc"])
    
    zi_stats["nb_zero_prop"]=np.nan
    #fill out valid nb cases with zero proportion...
    zi_stats.loc[zi_stats["valid_nb"],"nb_zero_prop"]=zi_stats["p"]**zi_stats["r"]
    #fill out non-valid nb cases with zero portion using poisson
    zi_stats.loc[~zi_stats["valid_nb"],"nb_zero_prop"]=np.exp(-zi_stats["mean_umis_mpra_bc"])

    assert ~any(zi_stats["nb_zero_prop"].isna())


    zi_stats["zero_inflation"]=zi_stats["gross_zero_prop"]-zi_stats["nb_zero_prop"]
    zi_stats["zero_inflation"]=np.clip(zi_stats["zero_inflation"],0,1)
    
    zi_stats=zi_stats.groupby("rep_id")["zero_inflation"].mean()

    # now let's reindex
    zi_stats=zi_stats.reindex(indicies["zi_idx"])
    #logistic function
    zi_beta=1/(1 + np.exp(-zi_stats.to_numpy()))

    #now calculate nb betas
    working_nb=nb_stats.copy()
    ref=working_nb.loc["reference"]["mean_umis_mpra_bc"]
    working_nb["fc"]=working_nb["mean_umis_mpra_bc"]/ref
    working_nb["lfc"]=np.log(working_nb["fc"])
    working_nb["beta"]=working_nb["lfc"]
    working_nb["beta"].loc["reference"]=np.log(working_nb["mean_umis_mpra_bc"].loc["reference"])
    
    nb_betas=working_nb["beta"]

    
    #now calculate theta betas
    working_nb=nb_stats.copy()
    thetas=working_nb["mean_umis_mpra_bc"]**2/(working_nb["var_umis_mpra_bc"]-working_nb["mean_umis_mpra_bc"])
    thetas=thetas[working_nb["valid_nb"]]
    beta_theta=np.log(np.mean(thetas))

    #with our table constructed, we can now extract our actual parameter estimates
    #first, the betas for the nb portion

    nb_betas=nb_betas.reindex(indicies["nb_idx"])

    init={}
    init["x_mu"]=nb_betas.to_numpy()
    init["x_pi"]=zi_beta
    init["theta"]=beta_theta


    return init

_mom_from_training_data(data=dat,split="cell_type",subset="Cardiomyocytes",indicies=_matricies_to_order(matricies=mats))
#_matricies_to_order(matricies=mats)

{'x_mu': array([-3.27615119,  8.32132871,  7.62087508,  6.38558224,  7.83289014,
         8.75128315,  8.96056099,  8.77144653,  7.38464493,  9.04435621,
         6.78957834,  8.5192327 ,  6.48276099,  8.58678165,  8.58728302,
         6.35183846,  7.22329883,  8.07982089,  8.52787863,  9.34954089,
         6.65366317,  7.68242882,  9.06691145,  8.75055816,  8.74186044,
         8.91563564,  8.75818241,  8.92174444,  8.67980874,  8.97265814,
         8.77458022,  6.33857484,  4.95297373,  7.9143745 ,  9.08134501,
         8.62725437,  8.95531093,  8.40864267,  9.07073346,  8.56635684,
         6.54100379,  9.04365582,  8.80685994,  9.07391656,  8.0823701 ,
         7.93004351,  6.29479627,  9.0257079 ,  9.22662428,  7.0816041 ,
         8.56954361, -0.2830854 ,  0.01805465, -0.7949085 , -0.72479802,
        -0.89010595, -0.32287193, -0.59920783, -0.64582215, -0.51652506,
        -0.71775897, -0.50661698, -0.94335651, -0.78141412, -1.26359119,
        -0.5305113 , -0.76940321, -0.349401

manually calc reference beta...

In [8]:
refmean=dat[(dat["cell_type"]=="Cardiomyocytes") & (dat["cre_id"]=="reference")]["umis_mpra_bc"].mean()

In [9]:
refmean

0.03777335984095427

$$e^{\beta_{ref}}=E[ref]\tag{2}$$

In [10]:
np.log(refmean)

-3.2761511909332985